<a href="https://colab.research.google.com/github/LP-D/claude/blob/main/notebooks/final_campaign/VIX_FINAL_ML_SCAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# VIX Final ML Scan v1 — grille complète, résumable (2/4)

**Rôle.** Deuxième des 4 notebooks de la campagne finale. Recharge le dataset produit par `VIX_FINAL_FEATURES` (features causales + interactions inter-tickers) et lance le scan walk-forward complet demandé :

$$6\ \text{horizons} \times 4\ \text{régimes} \times 11\ N(5..15) \times 5\ \text{samplers} \times 5\ \text{algos (dont CatBoost)} \times 5\ \text{folds} \approx 66\,000\ \text{entraînements}.$$

**Résumable, indispensable vu le volume.** À ce rythme, le scan prend probablement plusieurs dizaines d'heures — bien au-delà d'une session Colab. Ce notebook gère ça explicitement :
- Chaque combinaison terminée est écrite immédiatement dans `vix_final_ml_scan_results.csv`.
- **Au démarrage**, la progression déjà poussée sur GitHub (sessions précédentes) est récupérée automatiquement — on ne repart jamais de zéro.
- **Pendant le run**, la progression est repoussée périodiquement (tous les 1000 résultats) — une déconnexion Colab ne fait perdre que le travail depuis le dernier checkpoint, pas tout le run.
- **Pour continuer** : rouvre simplement ce notebook et relance toutes les cellules. Il détecte automatiquement ce qui est déjà fait.

**Prérequis** : `VIX_FINAL_FEATURES.ipynb` doit avoir tourné et poussé son dataset (secret Colab `GITHUB_TOKEN` requis, ici aussi, pour la reprise/le partage de progression).

**Comparateurs déjà établis** (walk-forward, notebooks précédents) : GLOBAL RandomForest h=5j (F1_dir≈0.610±0.025, F1_UP_FORT≈0.359), TFT h=5j (F1_dir≈0.559, F1_UP_FORT≈0.118 — infirmé).


In [ ]:
import subprocess, sys
pkgs = ['xgboost','lightgbm','catboost','shap','xlsxwriter','imbalanced-learn','pyarrow']
subprocess.run([sys.executable,'-m','pip','install','-q']+pkgs, check=False)
print("Installation OK")


In [ ]:
import os, time, json, warnings, random
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import shap
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import f1_score, accuracy_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from imblearn.over_sampling import SMOTE, BorderlineSMOTE, ADASYN
from imblearn.combine import SMOTETomek, SMOTEENN

SEED = 42; random.seed(SEED); np.random.seed(SEED)

NOTEBOOK_NAME = 'VIX_FINAL_ML_SCAN'
NOTEBOOK_VERSION = 'v1'

CONFIG = {
    'flat_thr': 0.003,
    'horizons': [1, 2, 3, 5, 7, 10],
    'regimes': ['GLOBAL', 'CALM', 'NORMAL', 'STRESS'],
    'n_features_grid': list(range(5, 16)),
    'samplers': ['SMOTE', 'BorderlineSMOTE', 'ADASYN', 'SMOTETomek', 'SMOTEENN'],
    'algos': ['XGBoost', 'LightGBM', 'RandomForest', 'GradientBoosting', 'CatBoost'],
    'n_wf_folds': 5,
    'min_train_frac': 0.40,   # doit matcher VIX_FINAL_FEATURES pour retrouver les mêmes folds
    'shap_sample': 500,
    'pool_prefilter': 450,
    'min_train_rows': 100, 'min_test_rows': 20,
}
TARGET_COL = 'VIX_Amplitude_Class'
RESULTS_CSV = 'vix_final_ml_scan_results.csv'
GITHUB_REPO = 'LP-D/claude'
FEATURES_BRANCH = 'results/vix-final-features'
RESULTS_BRANCH = 'results/vix-final-ml-scan'

n_combos = (len(CONFIG['horizons']) * len(CONFIG['regimes']) * len(CONFIG['n_features_grid'])
            * len(CONFIG['samplers']) * len(CONFIG['algos']) * CONFIG['n_wf_folds'])
print(f"{NOTEBOOK_NAME} {NOTEBOOK_VERSION} | grille max = {n_combos} combinaisons "
      f"({len(CONFIG['horizons'])}h × {len(CONFIG['regimes'])}reg × {len(CONFIG['n_features_grid'])}N × "
      f"{len(CONFIG['samplers'])}samplers × {len(CONFIG['algos'])}algos × {CONFIG['n_wf_folds']}folds)")


In [ ]:
# ============================================================
# CHARGEMENT DU DATASET PARTAGÉ (produit par VIX_FINAL_FEATURES)
# ============================================================
import subprocess

try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN')

if not os.path.exists('vix_final_features.parquet'):
    auth = f"{GITHUB_TOKEN}@" if GITHUB_TOKEN else ""
    url = f"https://{auth}github.com/{GITHUB_REPO}.git"
    workdir = "/content/_vix_features_pull"
    subprocess.run(["rm", "-rf", workdir], check=False)
    clone = subprocess.run(["git", "clone", "--depth", "1", "--branch", FEATURES_BRANCH, url, workdir],
                           capture_output=True, text=True)
    if clone.returncode != 0:
        raise RuntimeError(
            "Impossible de récupérer le dataset partagé depuis "
            f"'{FEATURES_BRANCH}'. As-tu bien exécuté VIX_FINAL_FEATURES.ipynb en premier "
            f"(et poussé son résultat) ? Détail: {clone.stderr[-500:]}")
    subprocess.run(["cp", f"{workdir}/vix_final_features.parquet", "."], check=True)
    subprocess.run(["cp", f"{workdir}/vix_final_features_meta.json", "."], check=True)
    print(f"[PULL OK] Dataset récupéré depuis '{FEATURES_BRANCH}'")
else:
    print("[SKIP] vix_final_features.parquet déjà présent localement")

df_features = pd.read_parquet('vix_final_features.parquet')
with open('vix_final_features_meta.json') as f:
    meta = json.load(f)
FEATURE_POOL = meta['feature_pool']
VIX_COL = meta['vix_col']; SPX_COL = meta['spx_col']
print(f"Dataset: {df_features.shape} | VIX={VIX_COL} | pool: {len(FEATURE_POOL)} features "
      f"(dont {len(meta['interaction_features'])} interactions) | source: {meta['date_min']} → {meta['date_max']}")

all_dates = df_features.dropna(how='all').index.sort_values()
n_obs = len(all_dates)
first_cut = int(n_obs * CONFIG['min_train_frac'])
test_span = (n_obs - first_cut) // CONFIG['n_wf_folds']
FOLD_CUTS = [first_cut + k * test_span for k in range(CONFIG['n_wf_folds'] + 1)]
FOLD_CUTS[-1] = n_obs
for k in range(CONFIG['n_wf_folds']):
    print(f"  Fold {k+1}: train → {all_dates[FOLD_CUTS[k]-1].date()} | "
          f"test {all_dates[FOLD_CUTS[k]].date()} → {all_dates[FOLD_CUTS[k+1]-1].date()}")


In [ ]:
def build_target(vix_series, horizon, split_idx):
    vix = vix_series.ffill().bfill(); vix_tr = vix.iloc[:split_idx]
    calm_thr = vix_tr.quantile(0.33); stress_thr = vix_tr.quantile(0.67)
    regime = pd.Series('NORMAL', index=vix.index)
    regime[vix < calm_thr] = 'CALM'; regime[vix >= stress_thr] = 'STRESS'
    ret = (vix.shift(-horizon) / vix) - 1
    flat = ret.abs() < CONFIG['flat_thr']
    ret = ret.loc[~flat].dropna(); reg_r = regime.reindex(ret.index)
    cut_date = vix.index[min(split_idx, len(vix) - 1)]
    ret_tr = ret.loc[ret.index < cut_date]; reg_tr = reg_r.loc[ret_tr.index]
    thr = {}
    for reg in ['CALM', 'NORMAL', 'STRESS']:
        sub = ret_tr[reg_tr == reg]
        thr[reg] = (sub.quantile(0.25) if len(sub) >= 20 else ret_tr.quantile(0.25),
                    sub.quantile(0.75) if len(sub) >= 20 else ret_tr.quantile(0.75))
    thr['GLOBAL'] = (ret_tr.quantile(0.25), ret_tr.quantile(0.75))
    def classify(r, reg):
        q25, q75 = thr.get(reg, (0, 0))
        if r < q25: return 0
        if r < 0:   return 1
        if r < q75: return 2
        return 3
    target = pd.Series([classify(r, reg_r[i]) for i, r in ret.items()], index=ret.index, name=TARGET_COL)
    return target, reg_r, thr

def metrics(y_true, y_pred):
    # [FIX] CatBoostClassifier.predict() renvoie un tableau 2D (n,1) pour le
    # multiclasse (contrairement à la convention scikit-learn 1D des autres
    # algos) -> itérer dessus donne des sous-tableaux de longueur 1, non
    # hashables comme clé de dict ('unhashable type: numpy.ndarray'), ce qui
    # faisait échouer CatBoost à 100% et corrompait le CSV (colonne 'error'
    # en plus -> lignes de largeur variable, cf. 05_scan_engine.py).
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    dm = {0: 'DOWN', 1: 'DOWN', 2: 'UP', 3: 'UP'}
    yd_t = [dm[y] for y in y_true]; yd_p = [dm[y] for y in y_pred]
    m = {'F1_4cls': round(f1_score(y_true, y_pred, average='macro', zero_division=0), 4),
         'Acc_dir': round(accuracy_score(yd_t, yd_p), 4),
         'F1_dir': round(f1_score(yd_t, yd_p, average='macro', zero_division=0), 4)}
    ui = [i for i, y in enumerate(y_true) if dm[y] == 'UP']
    di = [i for i, y in enumerate(y_true) if dm[y] == 'DOWN']
    if len(ui) >= 10:
        yt = ['FORT' if y_true[i] == 3 else 'FAIBLE' for i in ui]
        yp = ['FORT' if y_pred[i] == 3 else 'FAIBLE' for i in ui]
        m['F1_UP_FORT'] = round(f1_score(yt, yp, pos_label='FORT', average='binary', zero_division=0), 4)
    else: m['F1_UP_FORT'] = np.nan
    if len(di) >= 10:
        yt = ['FORT' if y_true[i] == 0 else 'FAIBLE' for i in di]
        yp = ['FORT' if y_pred[i] == 0 else 'FAIBLE' for i in di]
        m['F1_DOWN_FORT'] = round(f1_score(yt, yp, pos_label='FORT', average='binary', zero_division=0), 4)
    else: m['F1_DOWN_FORT'] = np.nan
    return m

def get_clf(algo):
    if algo == 'XGBoost':
        return XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05, subsample=0.8,
                             colsample_bytree=0.8, min_child_weight=3, eval_metric='mlogloss',
                             objective='multi:softprob', random_state=SEED, n_jobs=-1, verbosity=0)
    if algo == 'LightGBM':
        return LGBMClassifier(n_estimators=200, max_depth=5, learning_rate=0.05, num_leaves=31,
                              min_child_samples=10, subsample=0.8, class_weight='balanced',
                              random_state=SEED, verbose=-1, n_jobs=-1)
    if algo == 'RandomForest':
        return RandomForestClassifier(n_estimators=200, max_depth=6, min_samples_leaf=5,
                                      class_weight='balanced', random_state=SEED, n_jobs=-1)
    if algo == 'GradientBoosting':
        return GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=4,
                                          min_samples_leaf=10, subsample=0.8, random_state=SEED)
    if algo == 'CatBoost':
        return CatBoostClassifier(iterations=200, depth=6, learning_rate=0.05, loss_function='MultiClass',
                                  auto_class_weights='Balanced', random_state=SEED, verbose=False,
                                  allow_writing_files=False)
    raise ValueError(algo)

def get_samp(name):
    return {'SMOTE': SMOTE(random_state=SEED),
            'BorderlineSMOTE': BorderlineSMOTE(random_state=SEED, kind='borderline-1'),
            'ADASYN': ADASYN(random_state=SEED),
            'SMOTETomek': SMOTETomek(random_state=SEED),
            'SMOTEENN': SMOTEENN(random_state=SEED)}[name]

def shap_rank(X_tr, y_tr, pool_names, top_n, prefilter):
    nf = X_tr.shape[1]
    if nf > prefilter:
        pf = XGBClassifier(n_estimators=60, max_depth=4, learning_rate=0.1, objective='multi:softprob',
                           eval_metric='mlogloss', random_state=SEED, n_jobs=-1, verbosity=0)
        pf.fit(X_tr, y_tr); keep = np.argsort(pf.feature_importances_)[::-1][:prefilter]
    else:
        keep = np.arange(nf)
    Xk = X_tr[:, keep]
    pilot = XGBClassifier(n_estimators=80, max_depth=4, learning_rate=0.1, objective='multi:softprob',
                          eval_metric='mlogloss', random_state=SEED, n_jobs=-1, verbosity=0)
    pilot.fit(Xk, y_tr)
    sv = np.abs(np.array(shap.TreeExplainer(pilot).shap_values(Xk[:min(CONFIG['shap_sample'], len(Xk))])))
    nfk = Xk.shape[1]
    feat_axes = [ax for ax in range(sv.ndim) if sv.shape[ax] == nfk]
    if len(feat_axes) == 1:
        arr = sv.mean(axis=tuple(ax for ax in range(sv.ndim) if ax != feat_axes[0]))
    else:
        arr = np.asarray(pilot.feature_importances_)
    order = np.argsort(np.asarray(arr).ravel())[::-1][:top_n]
    return list(keep[order])

print("Helpers OK (build_target, metrics, get_clf x5, get_samp x5, shap_rank)")


In [ ]:
# ============================================================
# SYNCHRONISATION DE LA PROGRESSION AVEC GITHUB
# [IMPORTANT] Le disque Colab est éphémère d'une session à l'autre (nouvelle
# VM à chaque reconnexion) : RESULTS_CSV local ne survit PAS à une
# déconnexion. Pour reprendre après une coupure (quasi certaine vu la durée
# du scan), il faut (1) récupérer la progression déjà poussée AVANT de
# recalculer done_keys, et (2) republier régulièrement PENDANT le run, pas
# seulement à la fin — sinon un plantage juste avant la fin ferait tout
# perdre.
# ============================================================
import subprocess

try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN')

_PUSH_WORKDIR = "/content/_vix_ml_scan_push"

def pull_progress():
    """Récupère RESULTS_CSV déjà poussé (sessions précédentes), s'il existe."""
    if os.path.exists(RESULTS_CSV):
        print(f"[SKIP PULL] {RESULTS_CSV} déjà présent localement.")
        return
    if not GITHUB_TOKEN:
        print("[SKIP PULL] Pas de GITHUB_TOKEN — impossible de vérifier une progression antérieure poussée. "
              "Le scan démarre de zéro dans ce runtime.")
        return
    url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"
    exists = subprocess.run(["git", "ls-remote", "--exit-code", "--heads", url, RESULTS_BRANCH],
                            capture_output=True, text=True)
    if exists.returncode != 0:
        print(f"[INFO] Aucune progression antérieure sur '{RESULTS_BRANCH}' — nouveau run.")
        return
    workdir = "/content/_vix_ml_scan_pull"
    subprocess.run(["rm", "-rf", workdir], check=False)
    clone = subprocess.run(["git", "clone", "--depth", "1", "--branch", RESULTS_BRANCH, url, workdir],
                           capture_output=True, text=True)
    if clone.returncode == 0 and os.path.exists(f"{workdir}/{RESULTS_CSV}"):
        subprocess.run(["cp", f"{workdir}/{RESULTS_CSV}", "."], check=True)
        n = sum(1 for _ in open(RESULTS_CSV)) - 1
        print(f"[PULL OK] Progression antérieure récupérée : ~{n} lignes déjà faites.")
    else:
        print(f"[WARN] Branche '{RESULTS_BRANCH}' trouvée mais {RESULTS_CSV} absent — nouveau run.")

def push_progress(label=''):
    """Publie l'état courant de RESULTS_CSV (à appeler périodiquement + en fin de run)."""
    if not GITHUB_TOKEN or not os.path.exists(RESULTS_CSV):
        return False
    try:
        subprocess.run(["rm", "-rf", _PUSH_WORKDIR], check=False)
        url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"
        clone = subprocess.run(["git", "clone", url, _PUSH_WORKDIR], capture_output=True, text=True)
        if clone.returncode != 0: return False
        exists = subprocess.run(["git", "-C", _PUSH_WORKDIR, "ls-remote", "--exit-code", "--heads",
                                  "origin", RESULTS_BRANCH], capture_output=True, text=True)
        if exists.returncode == 0:
            subprocess.run(["git", "-C", _PUSH_WORKDIR, "checkout", "-B", RESULTS_BRANCH,
                             f"origin/{RESULTS_BRANCH}"], check=True)
        else:
            subprocess.run(["git", "-C", _PUSH_WORKDIR, "checkout", "-B", RESULTS_BRANCH], check=True)
        subprocess.run(["cp", RESULTS_CSV, f"{_PUSH_WORKDIR}/{RESULTS_CSV}"], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "config", "user.email", "vix-colab@users.noreply.github.com"], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "config", "user.name", "VIX Final ML Scan Colab run"], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "add", RESULTS_CSV], check=True)
        commit = subprocess.run(["git", "-C", _PUSH_WORKDIR, "commit", "-m",
                                 f"Progression scan ML {label} — {pd.Timestamp.now():%Y-%m-%d %H:%M}"],
                                capture_output=True, text=True)
        if 'nothing to commit' in (commit.stdout or ''):
            return True
        push = subprocess.run(["git", "-C", _PUSH_WORKDIR, "push", "origin", RESULTS_BRANCH],
                              capture_output=True, text=True)
        ok = push.returncode == 0
        if ok: print(f"  [CHECKPOINT PUSHÉ] {label} ({pd.Timestamp.now():%H:%M:%S})")
        else: print(f"  [WARN push checkpoint] {push.stderr[-300:]}")
        return ok
    except Exception as e:
        print(f"  [WARN push checkpoint] {e}")
        return False

pull_progress()


In [ ]:
# ============================================================
# SCAN COMPLET CHECKPOINTÉ ET RÉSUMABLE
# horizon × fold × régime × N(5-15) × sampler × algo
# Chaque ligne terminée est ajoutée immédiatement à RESULTS_CSV. Au
# redémarrage (nouvelle session Colab après déconnexion), les combinaisons
# déjà présentes dans ce fichier sont sautées automatiquement — indispensable
# vu le volume (des dizaines d'heures, largement au-delà d'une session).
# ============================================================
KEY_COLS = ['horizon', 'regime', 'fold', 'N', 'sampler', 'algo']

done_keys = set()
if os.path.exists(RESULTS_CSV) and os.path.getsize(RESULTS_CSV) > 0:
    prev = pd.read_csv(RESULTS_CSV, usecols=KEY_COLS)
    done_keys = set(map(tuple, prev.values.tolist()))
    print(f"[REPRISE] {len(done_keys)}/{n_combos} combinaisons déjà faites — reprise en cours.")
else:
    print("[DÉMARRAGE] Aucun résultat existant — nouveau run.")

def already_done(h, reg, fold, N, samp, algo):
    return (h, reg, fold, N, samp, algo) in done_keys

def save_row(row):
    header = not (os.path.exists(RESULTS_CSV) and os.path.getsize(RESULTS_CSV) > 0)
    pd.DataFrame([row]).to_csv(RESULTS_CSV, mode='a', header=header, index=False)
    done_keys.add(tuple(row[c] for c in KEY_COLS))

t0 = time.time(); n_done_session = 0
for h in CONFIG['horizons']:
    for k in range(CONFIG['n_wf_folds']):
        cut, nxt = FOLD_CUTS[k], FOLD_CUTS[k + 1]
        cut_date, nxt_date = all_dates[cut], all_dates[nxt - 1]
        target, reg_r, _ = build_target(df_features[VIX_COL], h, cut)
        idx = target.index
        tr_mask_base = np.asarray(idx < cut_date)
        te_mask_base = np.asarray((idx >= cut_date) & (idx <= nxt_date))
        reg_al = reg_r.reindex(idx).fillna('NORMAL').values

        for reg in CONFIG['regimes']:
            # tout ce qui reste à faire pour cette clé (h,reg,fold) ?
            remaining = [(N, s, a) for N in CONFIG['n_features_grid']
                         for s in CONFIG['samplers'] for a in CONFIG['algos']
                         if not already_done(h, reg, k + 1, N, s, a)]
            if not remaining:
                continue

            tr_mask = tr_mask_base & (reg_al == reg if reg != 'GLOBAL' else True)
            te_mask = te_mask_base & (reg_al == reg if reg != 'GLOBAL' else True)
            y_tr = target.values[tr_mask].astype(int); y_te = target.values[te_mask].astype(int)
            if len(y_tr) < CONFIG['min_train_rows'] or len(y_te) < CONFIG['min_test_rows']:
                continue

            X_pool = df_features[FEATURE_POOL].reindex(idx)
            sc = RobustScaler()
            X_tr = sc.fit_transform(np.nan_to_num(X_pool.values[tr_mask]))
            X_te = sc.transform(np.nan_to_num(X_pool.values[te_mask]))
            N_max = CONFIG['n_features_grid'][-1]
            ranked_idx = shap_rank(X_tr, y_tr, FEATURE_POOL, N_max, CONFIG['pool_prefilter'])

            for N in CONFIG['n_features_grid']:
                needed_algos_by_sampler = {
                    s: [a for a in CONFIG['algos'] if not already_done(h, reg, k + 1, N, s, a)]
                    for s in CONFIG['samplers']}
                if not any(needed_algos_by_sampler.values()):
                    continue
                cols = ranked_idx[:N]
                Xtr_n, Xte_n = X_tr[:, cols], X_te[:, cols]

                for s in CONFIG['samplers']:
                    algos_todo = needed_algos_by_sampler[s]
                    if not algos_todo:
                        continue
                    try:
                        Xr, yr = get_samp(s).fit_resample(Xtr_n, y_tr)
                    except Exception:
                        Xr, yr = Xtr_n, y_tr
                    for a in algos_todo:
                        try:
                            clf = get_clf(a); clf.fit(Xr, yr)
                            met = metrics(y_te, clf.predict(Xte_n))
                        except Exception as e:
                            # [FIX] ne jamais ajouter de clé supplémentaire au dict sauvegardé :
                            # une colonne en plus ('error') rend les lignes de largeur variable
                            # et casse pd.read_csv pour tous les lecteurs en aval (VIX_FINAL_OPTUNA
                            # notamment). L'erreur est seulement affichée, pas persistée.
                            met = {'F1_4cls': np.nan, 'Acc_dir': np.nan, 'F1_dir': np.nan,
                                   'F1_UP_FORT': np.nan, 'F1_DOWN_FORT': np.nan}
                            print(f"  [WARN] {h}j {reg} fold{k+1} N={N} {s} {a}: {str(e)[:150]}")
                        save_row({'horizon': h, 'regime': reg, 'fold': k + 1, 'N': N, 'sampler': s, 'algo': a,
                                  'n_train': len(y_tr), 'n_test': len(y_te),
                                  'test_start': str(cut_date.date()), 'test_end': str(nxt_date.date()), **met})
                        n_done_session += 1
                        if n_done_session % 200 == 0:
                            elapsed = time.time() - t0
                            rate = n_done_session / elapsed
                            remaining_n = n_combos - len(done_keys)
                            eta_h = (remaining_n / rate) / 3600 if rate > 0 else float('nan')
                            print(f"  [{len(done_keys)}/{n_combos}] session={n_done_session} "
                                  f"h={h}j {reg} N={N} {s} {a} F1_dir={met.get('F1_dir')} "
                                  f"| {elapsed/60:.1f}min écoulées, ETA restante≈{eta_h:.1f}h")
                        # checkpoint périodique : publie la progression pour survivre à une
                        # déconnexion Colab (le disque local ne survit pas d'une session à l'autre)
                        if n_done_session % 1000 == 0:
                            push_progress(label=f"{len(done_keys)}/{n_combos}")

push_progress(label=f"fin de session ({len(done_keys)}/{n_combos})")
print(f"\n[SCAN] {len(done_keys)}/{n_combos} combinaisons au total "
      f"({n_done_session} faites cette session, {(time.time()-t0)/60:.1f}min)")


In [ ]:
# ============================================================
# SYNTHÈSE (à date — peut être relancée à tout moment, même run partiel)
# ============================================================
df_scan = pd.read_csv(RESULTS_CSV) if os.path.exists(RESULTS_CSV) else pd.DataFrame()
print(f"Progression: {len(df_scan)}/{n_combos} lignes ({len(df_scan)/max(n_combos,1):.1%})")

if len(df_scan):
    agg = (df_scan.dropna(subset=['F1_dir'])
           .groupby(['horizon', 'regime', 'N', 'sampler', 'algo'])
           .agg(F1_dir_mean=('F1_dir', 'mean'), F1_dir_std=('F1_dir', 'std'),
                F1_UP_FORT_mean=('F1_UP_FORT', 'mean'), F1_DOWN_FORT_mean=('F1_DOWN_FORT', 'mean'),
                n_folds=('fold', 'nunique'))
           .reset_index())
    agg = agg[agg['n_folds'] >= 3].round(4)

    print("\n### TOP 15 par F1_dir moyen (toutes configs confondues) ###")
    print(agg.sort_values('F1_dir_mean', ascending=False).head(15).to_string(index=False))
    print("\n### TOP 15 par F1_UP_FORT moyen ###")
    print(agg.sort_values('F1_UP_FORT_mean', ascending=False).head(15).to_string(index=False))
    triple = agg[(agg['F1_dir_mean'] > 0.50) & (agg['F1_UP_FORT_mean'] > 0.50) & (agg['F1_DOWN_FORT_mean'] > 0.50)]
    print(f"\n### Configs > 0.50 sur les 3 métriques simultanément : {len(triple)} ###")
    if len(triple): print(triple.sort_values('F1_dir_mean', ascending=False).head(15).to_string(index=False))

    print("\nRéférence établie (GLOBAL RandomForest h=5j, walk-forward, notebooks précédents) : "
          "F1_dir≈0.610±0.025, F1_UP_FORT≈0.359, F1_DOWN_FORT≈0.627")

    try:
        with pd.ExcelWriter('VIX_FINAL_ML_SCAN_report.xlsx', engine='xlsxwriter') as w:
            agg.sort_values('F1_dir_mean', ascending=False).to_excel(w, 'Agg_by_config', index=False)
            agg.sort_values('F1_UP_FORT_mean', ascending=False).head(50).to_excel(w, 'Top_UP_FORT', index=False)
            triple.to_excel(w, 'Triple_gt_0.50', index=False)
        print("\n[SAVE] VIX_FINAL_ML_SCAN_report.xlsx (snapshot à date)")
    except Exception as e:
        print(f"[WARN Export] {e}")
else:
    print("Aucun résultat pour l'instant.")
print(f"\n[NOTE] {RESULTS_CSV} contient le détail complet — le recharger pour reprendre le scan.")


In [ ]:
# ============================================================
# PUSH FINAL DU RAPPORT (xlsx) EN PLUS DU CSV DE PROGRESSION
# La cellule de scan pousse déjà RESULTS_CSV périodiquement ; celle-ci ajoute
# le rapport agrégé xlsx (utile pour une relecture rapide sans recharger le CSV).
# ============================================================
def push_report_file():
    if not GITHUB_TOKEN or not os.path.exists('VIX_FINAL_ML_SCAN_report.xlsx'):
        print("[SKIP] Pas de token ou pas de rapport à pousser.")
        return
    try:
        subprocess.run(["cp", "VIX_FINAL_ML_SCAN_report.xlsx", f"{_PUSH_WORKDIR}/VIX_FINAL_ML_SCAN_report.xlsx"], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "add", "VIX_FINAL_ML_SCAN_report.xlsx"], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "commit", "-m",
                       f"Rapport agrégé — {pd.Timestamp.now():%Y-%m-%d %H:%M}"],
                       capture_output=True, text=True)
        push = subprocess.run(["git", "-C", _PUSH_WORKDIR, "push", "origin", RESULTS_BRANCH],
                              capture_output=True, text=True)
        if push.returncode == 0:
            print(f"[PUSH OK] VIX_FINAL_ML_SCAN_report.xlsx sur '{RESULTS_BRANCH}'")
        else:
            print(f"[WARN] {push.stderr[-300:]}")
    except Exception as e:
        print(f"[WARN] {e}")

push_progress(label='rapport final')  # s'assure que le workdir/branche existent
push_report_file()
